In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/sittacanada@gmail.com/consolidated_pipeline/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "products", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportbar-dp-30/{data_source}/*.csv'
print(base_path)

In [0]:
df = (
    spark.read.format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .select("*", "_metadata.file_name", "_metadata.file_size")
)

In [0]:
df.printSchema()

In [0]:
display(df.limit(10))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")

## Silver

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source};")
df_bronze.show(10)

### drop duplicates

In [0]:
print('Rows before duplicates dropped:', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after duplicates dropped:', df_silver.count())

### title case fix

In [0]:
df_silver.select('category').distinct().show()

In [0]:
# case fix
df_silver = df_silver.withColumn(
    "category",
    F.when(F.col("category").isNull(), None)
    .otherwise(F.initcap("category"))
)

In [0]:
df_silver.select('category').distinct().show()

In [0]:
# spelling mistak

df_silver = df_silver.withColumn(
    "category",
    F.regexp_replace(F.col("category"), "(?i)Protien", "Protein")
)

In [0]:
display(df_silver.limit(5))

### standardizing the tables

In [0]:
df_silver = (
    df_silver
        .withColumn(
            "division",
            F
            .when(F.col("category")=="Energy Bars", "Nutrition Bars")
            .when(F.col("category")=="Protein Bars", "Nutrition Bars")
            .when(F.col("category")=="Gronola & Cereals", "Breakfast Foods")
            .when(F.col("category")=="Recovery Dairy", "Dairy & Recovery")
            .when(F.col("category")=="Halthy Snacks", "Healthy Snacks")
            .when(F.col("category")=="Electrolyte Mix", "Hydration & Eletrolytes")
            .otherwise("Other")
        )
)


df_silver = df_silver.withColumn(
    "variant",
    F.regexp_extract(F.col("product_name"), r"\((.*?)\)", 1)
)

df_silver = (
    df_silver
    # generate deterministic prodcut_code from product_name
    .withColumn(
        "product_code",
        F.sha2(F.col("product_name").cast("string"), 256)
    )

    # clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        "product_id",
        F.when(
            F.col("product_id").cast("string").rlike("^[0-9]+$"),
            F.col("product_id").cast("string")
        ).otherwise(F.lit(999999).cast("string"))
    )
    # rename product_name -> product
    .withColumnRenamed("product_name", "product")
)


In [0]:
display(df_silver)

In [0]:
df_silver = df_silver.select("product_code", "division", "category", "product", "variant", "product_id", "read_timestamp", "file_name", "file_size")

In [0]:
display(df_silver)

In [0]:
df_silver.write\
    .format("delta")\
    .option("mergeSchema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

spark.sql(f"""
ALTER TABLE {catalog}.{silver_schema}.{data_source}
SET TBLPROPERTIES (
  delta.enableChangeDataFeed = true
)
""")

## Gold

In [0]:
df_silver = spark.sql(f"SELECT * FROM {catalog}.{silver_schema}.{data_source};")
df_gold = df_silver.select("product_code", "product_id", "division", "category", "product", "variant")
df_gold.show(5)


In [0]:
df_gold.write\
    .format("delta")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

### Merging Data source with parent

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_products")
df_child_products = spark.sql(f"SELECT product_code, division, category, product, variant FROM fmcg.gold.sb_dim_products;")
df_child_products.show(5)

In [0]:
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.product_code = source.product_code"
).whenMatchedUpdate(
    set={
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).whenNotMatchedInsert(
    values={
        "product_code": "source.product_code",
        "division": "source.division",
        "category": "source.category",
        "product": "source.product",
        "variant": "source.variant"
    }
).execute()

